# MATH840 — Practice 1: Onboarding

**Week 1 | Time Series | Kyiv School of Economics**

This session is **not graded**. Nothing you produce today counts towards your 100 points.

That is deliberate. From Week 2, every practice session ends with a submission that *does*
count, and the session is 80 minutes long. There is no time in Week 2 to discover that your
environment is broken, that you cannot export a PDF, or that Moodle rejects your file.

So today we break all of that on purpose, while it is free.

---

### By 12:50 today, five things must be true

1. You can run Python with the course libraries.
2. You know **which dataset is yours** for Weeks 2–3.
3. You have loaded it, and you know its frequency, its span, and whether its calendar is broken.
4. You have produced three plots: a time plot, a seasonal plot, and an ACF.
5. You have uploaded a test submission to Moodle: **one PDF and one source file**.

If all five are true, you are ready for the rest of the course. If any one is not, tell me
before you leave the room.

## 1. Environment

Run this first. It takes about a minute.

In [ ]:
!pip install -q statsforecast utilsforecast mlforecast plotly

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import hashlib

print("pandas", pd.__version__)
print("numpy ", np.__version__)

## 2. Get your dataset

Your dataset for Weeks 2–3 is determined by **your student identifier** — the one you use in
Moodle. Not your name, not your email: the identifier, exactly as it appears in your profile.

The same mechanism issues your personal forecasting series in Week 4. Today it is a dress
rehearsal on data that costs you nothing.

⚠️ **Write your identifier down somewhere you will still have it in Week 4.** The same string
has to produce the same series then.

In [ ]:
BASE = ("https://raw.githubusercontent.com/Aranaur/aranaur.rbind.io/"
        "main/lectures/kse/MATH840/26autumn/data")

POOL = (
    # Australian quarterly production, 1956-2010
    [{"file": "aus_production.csv", "column": c, "freq": "QS", "season": 4,
      "label": f"Australian production: {c}"}
     for c in ["Beer", "Tobacco", "Bricks", "Cement", "Electricity", "Gas"]]
    # Australian domestic tourism, quarterly overnight trips by state and purpose
    + [{"file": "tourism.csv", "state": s, "purpose": p, "freq": "QS", "season": 4,
        "label": f"Tourism: {p} trips to {s}"}
       for s in ["South Australia", "Northern Territory", "Western Australia",
                 "Victoria", "New South Wales", "Queensland", "ACT", "Tasmania"]
       for p in ["Business", "Holiday", "Other", "Visiting"]]
    # Ansett Airlines, weekly passengers by route and class
    + [{"file": "ansett.csv", "route": r, "class": c, "freq": "W", "season": 52,
        "label": f"Ansett passengers: {r}, {c} class"}
       for r in ["MEL-SYD", "MEL-ADL", "SYD-BNE", "MEL-BNE", "ADL-PER", "MEL-PER"]
       for c in ["Business", "Economy"]]
)


def assign_dataset(student_id: str) -> dict:
    """Deterministic, reproducible, and identical on every machine.

    Note: Python's built-in hash() is randomised per process and would give you a
    different dataset on every run. hashlib is not.
    """
    digest = hashlib.sha256(student_id.strip().encode("utf-8")).hexdigest()
    return POOL[int(digest, 16) % len(POOL)]

In [ ]:
MY_ID = ""   # <-- put your student identifier here, between the quotes

if not MY_ID.strip():
    raise ValueError("Set MY_ID to your student identifier before running the rest.")

mine = assign_dataset(MY_ID)
print(mine["label"])
mine

Now load it.

In [ ]:
def load_my_series(spec: dict) -> pd.DataFrame:
    df = pd.read_csv(f"{BASE}/{spec['file']}")

    if spec["file"] == "aus_production.csv":
        out = df[["ds", spec["column"]]].rename(columns={spec["column"]: "y"})
    elif spec["file"] == "tourism.csv":
        sub = df[(df["State"] == spec["state"]) & (df["Purpose"] == spec["purpose"])]
        out = sub.groupby("ds", as_index=False)["y"].sum()
    else:  # ansett.csv
        sub = df[(df["Airports"] == spec["route"]) & (df["Class"] == spec["class"])]
        out = sub[["ds", "y"]].copy()

    out["ds"] = pd.to_datetime(out["ds"])
    return out.sort_values("ds").reset_index(drop=True)


y = load_my_series(mine)
y.head()

## 3. Interrogate the calendar

Before any plot, answer four questions about your series. In Week 2 this is worth a point on
the rubric; today it is practice.

- **Frequency** — what interval separates observations, and is it always the same one?
- **Gaps** — a missing period is not the same as a missing value. A missing value has a row; a gap does not.
- **Duplicates** — two rows claiming the same timestamp. Decide what they mean before you average them away.
- **Span** — how many complete seasonal cycles do you have? Fewer than three, and seasonal methods have very little to work with.

In [ ]:
print("rows:      ", len(y))
print("span:      ", y["ds"].min().date(), "→", y["ds"].max().date())
print("missing y: ", y["y"].isna().sum())
print("duplicates:", y["ds"].duplicated().sum())

gaps = y["ds"].diff().value_counts()
print("\nobserved spacings:")
print(gaps)

### Resolve duplicates first

Some series in the pool genuinely contain a repeated timestamp — and the two rows do **not**
agree on the value. That is not a bug in this template; it is what the data looks like. You
cannot put a series on a regular time grid until you have decided what a duplicate means.

There is no universally correct answer. If the two rows are two reports of the same week,
averaging them is reasonable. If they are two segments never meant to be merged, summing them
is. If one is a correction of the other, keeping the later one is. **The mark is for the
reasoning, not the function you called.**

In [ ]:
if y["ds"].duplicated().any():
    print("duplicated timestamps:")
    print(y[y["ds"].duplicated(keep=False)].sort_values("ds"))

    # Your decision. Sum? Mean? Keep the first? Keep the larger?
    # Whatever you choose, you must be able to defend it in one sentence.
    y = y.groupby("ds", as_index=False)["y"].mean()
    print(f"\nresolved by averaging → {len(y)} rows")
else:
    print("no duplicated timestamps")

### Put it on a regular grid

A gap becomes a row with a missing value — a problem you can see and decide about, instead of
one hiding in the index.

In [ ]:
grid = pd.date_range(y["ds"].min(), y["ds"].max(), freq=mine["freq"])
y_filled = y.set_index("ds").reindex(grid).rename_axis("ds").reset_index()

print(f"{len(y)} rows → {len(y_filled)} rows on a regular grid")
print(f"revealed gaps: {y_filled['y'].isna().sum() - y['y'].isna().sum()}")

The Nixtla stack has this built in — `from utilsforecast.preprocessing import fill_gaps`, then
`fill_gaps(y, freq=...)` on a frame with a `unique_id` column. Same result, and it handles many
series at once. We will use it from Week 2.

## 4. Three plots

This is the second half of today's lecture, applied to your own data.
[Chapter 2 of FPP](https://otexts.com/fpppy/) has all three.

### Time plot

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(y["ds"], y["y"], linewidth=1.2, color="#e64173")
ax.set_title(mine["label"])
ax.set_xlabel("")
ax.grid(alpha=0.3)
plt.show()

Look at it and name what you see: trend, seasonality, cycles, level shifts, outliers.
**Write the sentence down** — in Week 2 that sentence is part of the marked EDA.

### Seasonal plot

In [ ]:
d = y.dropna(subset=["y"]).copy()
d["year"] = d["ds"].dt.year
d["period"] = d["ds"].dt.quarter if mine["season"] == 4 else d["ds"].dt.isocalendar().week

fig, ax = plt.subplots(figsize=(12, 4))
for yr, grp in d.groupby("year"):
    ax.plot(grp["period"], grp["y"], marker="o", markersize=3, linewidth=1, label=yr)
ax.set_xlabel("period within year")
ax.set_title(f"Seasonal plot: {mine['label']}")
ax.grid(alpha=0.3)
plt.show()

### ACF

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

series = y.set_index("ds")["y"].dropna()
fig, ax = plt.subplots(figsize=(12, 4))
plot_acf(series, lags=min(40, len(series) // 3), ax=ax)
ax.set_title(f"ACF: {mine['label']}")
plt.show()

**Read your ACF before you leave:**

- Slow decay across many lags → **trend**.
- Peaks at multiples of the seasonal period → **seasonality**.
- Everything inside the blue band → indistinguishable from **white noise**, and nothing you fit will help.

## 5. The submission dry run

Every graded lab from Week 2 is submitted the same way. We do it once today, with nothing at stake.

### Produce a forecast — the dumbest one that exists

In [ ]:
H = 8  # horizon: 8 periods ahead

last_season = series.iloc[-mine["season"]:].values
forecast = np.array([last_season[i % mine["season"]] for i in range(H)])

future_ds = pd.date_range(series.index[-1], periods=H + 1, freq=mine["freq"])[1:]

submission = pd.DataFrame({"ds": future_ds, "forecast": forecast})
submission.to_csv("lab00_submission.csv", index=False)
submission

That is **seasonal naive** — the benchmark from the lecture, the one that finished 5th out of 19
last year. You have now implemented it, in four lines, before learning a single forecasting
method. Remember that when your ARIMA loses to it in Week 6.

### Download your files

In [ ]:
# In Colab this pops up a download. Locally it just prints a note.
try:
    from google.colab import files
    files.download("lab00_submission.csv")
except ImportError:
    print("Not in Colab — your CSV is saved next to this notebook.")

### Export and upload

1. Export this notebook to **PDF**: `File → Print → Save as PDF`.
2. Download the notebook itself: `File → Download → Download .ipynb`.
3. Upload **three files** to the Week 1 activity on Moodle:
   - `SURNAME_lab00.pdf`
   - `SURNAME_lab00.ipynb`
   - `SURNAME_lab00_submission.csv`

⚠️ If the PDF export fails — and for someone in the room it will — **fix it now**. From Week 2,
"I could not export the PDF" arrives after the deadline has passed, and the deadline is the end
of the session.

## 6. Before you leave

- [ ] Environment runs, libraries import.
- [ ] I know my dataset and I have recorded my student identifier.
- [ ] I can describe my series: frequency, span, gaps, duplicates.
- [ ] Three plots produced, and I can say in one sentence what my series does.
- [ ] Three files uploaded to Moodle.

## 7. Before next week

Read **Chapter 3** of [Forecasting: Principles and Practice, the Pythonic Way](https://otexts.com/fpppy/)
— transformations, decomposition, moving averages, STL.

Week 2 is your first graded lab: 80 minutes, same dataset, submitted before you leave the room.